|<img src="https://www.udp.cl/cms/wp-content/uploads/2021/06/UDP_LogoRGB_2lineas_Color_SinFondo.png" width="450" height="90">| <p><b>Ciencia de Datos para la Economía</b></p><p><b>Escuela de Ingeniería Comercial - Universidad Diego Portales</b></p><p><b>Profesor:</b> Luis Cuevas Parra</p><p><b>Modelos de Machine Learning:</b> Interpretación de Pronósticos</p>|
|---|---|

# Post-Lasso e interpretación de pronósticos

> **Objetivo de la sesión:** Interpretar un pronóstico Post-Lasso y comparar su precisión con Ridge, Lasso y Elastic Net.

Continuamos con los datos de U4-03, pero expresamos las variaciones de precios en **porcentaje mensual** para leer sus unidades sin anualizarlas. Usamos solo cuatro series y tres rezagos por serie para seguir cada paso.


## 0. Preparación del entorno de trabajo

Usamos datos mensuales de FRED, `scikit-learn` para la selección de variables y `statsmodels` para la regresión posterior a Lasso.


In [ ]:
# Bibliotecas usadas en U4-03 para datos, modelos y gráficos
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Datos: de niveles a variaciones mensuales

Pronosticamos la inflación total del mes $t$ con datos observados hasta el mes $t-1$.

| Serie FRED | Variable usada | Unidad |
|---|---|---|
| `CPIAUCSL` | Inflación total y sus rezagos | % mensual |
| `CPILFESL` | Inflación subyacente | % mensual |
| `WTISPLC` | Variación del precio del petróleo | % mensual |
| `FEDFUNDS` | Cambio de la tasa de política | Puntos porcentuales |

Para las tres primeras series calculamos $100[\log(P_t)-\log(P_{t-1})]$: una aproximación a la variación porcentual mensual. Para la tasa de política calculamos la diferencia mensual en puntos porcentuales. **No usamos niveles de precios ni anualizamos las variaciones.**

Cada predictor entra con uno, dos o tres meses de rezago. La muestra termina en septiembre de 2025 para mantener el período de U4-03. Los datos descargados hoy pueden contener revisiones posteriores a cada fecha histórica.


In [ ]:
# Descargar las mismas series FRED usadas en U4-03, pero solo cuatro de ellas
series_fred = ['CPIAUCSL', 'CPILFESL', 'WTISPLC', 'FEDFUNDS']

def descargar_fred(codigos):
    """Descarga desde FRED cada serie por separado y las une en un DataFrame."""
    dfs = []
    for codigo in codigos:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={codigo}"
        df_fred = pd.read_csv(url, index_col=0, parse_dates=True)[codigo]
        dfs.append(df_fred)
    return pd.concat(dfs, axis=1)

raw_data = descargar_fred(series_fred).loc['1985-01-01':'2025-09-01']
raw_data = raw_data.apply(pd.to_numeric, errors='coerce')

# Pasar índices de precios a variaciones; la tasa de política se diferencia
macro = pd.DataFrame(index=raw_data.index)
macro['inflacion'] = 100 * np.log(raw_data['CPIAUCSL']).diff()
macro['subyacente'] = 100 * np.log(raw_data['CPILFESL']).diff()
macro['petroleo'] = 100 * np.log(raw_data['WTISPLC']).diff()
macro['cambio_tasa'] = raw_data['FEDFUNDS'].diff()

# Usar solo los tres meses anteriores para pronosticar cada mes
lags = range(1, 4)
X_lags = pd.DataFrame(index=macro.index)
for col in macro.columns:
    for lag in lags:
        X_lags[f'{col}_lag{lag}'] = macro[col].shift(lag)

# Separar entrenamiento (1990-2004) y evaluación (desde 2005)
X_cols = list(X_lags.columns)
df = pd.concat([macro['inflacion'], X_lags], axis=1).dropna().loc['1990-01-01':]
inicio_oos = df.index.get_loc(pd.Timestamp('2005-01-01'))
train = df.iloc[:inicio_oos]
X_train = train[X_cols]
y_train = train['inflacion']

print(f'Entrenamiento: {len(train)} meses; evaluación: {len(df) - inicio_oos} meses')
display(df.head(3).round(2))


## 2. ¿Cómo se interpreta un coeficiente regularizado?

En una regresión lineal, un coeficiente describe cuánto cambia **el pronóstico del modelo** si un predictor aumenta una unidad y los demás se mantienen fijos. Esta lectura algebraica también sirve para Ridge, Lasso y Elastic Net, siempre que devolvamos los coeficientes a las unidades originales.

La penalización reduce el tamaño de los coeficientes para mejorar la predicción. Por ello, su magnitud **no es una estimación sin penalización** de la relación entre variables. Con predictores correlacionados, incluso el signo y el orden de los coeficientes pueden cambiar entre muestras o valores de la penalización. Un coeficiente cero en Lasso tampoco demuestra que la variable carezca de relación con la inflación.

Estimamos los tres métodos con la muestra inicial. Usamos el esquema de la sesión anterior: `StandardScaler` seguido de `RidgeCV`, `LassoCV` o `ElasticNetCV`, con cinco particiones temporales y las mismas grillas de penalización.


In [ ]:
# Mismas particiones temporales, grillas y modelos de U4-03
tscv = TimeSeriesSplit(n_splits=5)
grilla = np.logspace(-3, 2, 50)
modelos = {
    'Ridge': make_pipeline(StandardScaler(), RidgeCV(
        alphas=np.logspace(-2, 5, 50), cv=tscv)),
    'Lasso': make_pipeline(StandardScaler(), LassoCV(
        alphas=grilla, cv=tscv, max_iter=20000)),
    'Elastic Net': make_pipeline(StandardScaler(), ElasticNetCV(
        alphas=grilla, l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        cv=tscv, max_iter=20000)),
}

# Ajustar cada modelo solo con la muestra de entrenamiento
for modelo in modelos.values():
    modelo.fit(X_train, y_train)

# Mostrar la penalización elegida por validación cruzada
resumen = pd.DataFrame({
    'lambda elegido': [modelos[n][-1].alpha_ for n in modelos],
    'Mezcla L1': [0.0, 1.0, modelos['Elastic Net'][-1].l1_ratio_],
}, index=modelos.keys())
display(resumen.round(4))


,lambda elegido,Mezcla L1
Ridge,37.2759,0.0
Lasso,0.2223,1.0
Elastic Net,0.2223,0.9


### 2.1. Estandarización e interpretación

`StandardScaler` resta el promedio de cada predictor y divide por su desviación estándar: $z_j=(x_j-\bar{x}_j)/s_j$. La inflación que pronosticamos **no se estandariza**; permanece en porcentaje mensual.

Por eso, el coeficiente de Ridge, Lasso o Elastic Net indica cuánto cambia el **pronóstico de inflación mensual**, en puntos porcentuales, cuando un predictor sube **una desviación estándar** y los demás permanecen fijos.

La regularización es otra operación: penaliza coeficientes grandes para reducir el sobreajuste. Convertir un coeficiente a la unidad original del predictor cambia su forma de lectura, pero **no elimina la penalización**.


In [ ]:
# Coeficientes por una desviación estándar del predictor
coef_std = pd.DataFrame({nombre: modelo[-1].coef_ for nombre, modelo in modelos.items()},
                        index=X_cols)

# Dividir por la desviación estándar para volver a la unidad original
desv_x = pd.Series(modelos['Lasso'][0].scale_, index=X_cols)
coef_original = coef_std.div(desv_x, axis=0)

# Identificar las variables que Lasso dejó activas
seleccion_inicial = coef_std.index[coef_std['Lasso'].abs() > 1e-8].tolist()
print(f'Lasso seleccionó {len(seleccion_inicial)} de {len(X_cols)} predictores.')

print('Por una desviación estándar del predictor (pp de inflación mensual):')
display(coef_std.round(3))
print('Por una unidad original del predictor (pp de inflación mensual):')
display(coef_original.round(4))


## 3. Post-Lasso: seleccionar y volver a estimar

Post-Lasso usa las variables que Lasso dejó activas y vuelve a estimar sus coeficientes mediante MCO, sin penalización. El resultado depende de la selección previa: no garantiza coeficientes insesgados, efectos causales ni mejores pronósticos.

Si Lasso no selecciona ninguna variable, usamos como pronóstico la media de inflación del entrenamiento.


In [ ]:
# Ajustar MCO solo con las variables seleccionadas por Lasso
def ajustar_post_lasso(X, y, seleccion):
    if not seleccion:
        return None
    return sm.OLS(y, sm.add_constant(X[seleccion], has_constant='add')).fit()

post_inicial = ajustar_post_lasso(X_train, y_train, seleccion_inicial)

# Mostrar los coeficientes Post-Lasso en sus unidades originales
if post_inicial is not None:
    coef_post = post_inicial.params.drop('const')
else:
    coef_post = pd.Series(dtype=float)
display(coef_post.round(4).to_frame('Post-Lasso: por 1 unidad del predictor'))


Un coeficiente Post-Lasso describe cuánto cambia el pronóstico de inflación mensual, en puntos porcentuales, cuando ese predictor aumenta una unidad y los demás permanecen fijos. La unidad del predictor depende de la fila: para `petroleo_lag1` es un punto porcentual de variación mensual del petróleo; para `cambio_tasa_lag1` es un punto porcentual de cambio en la tasa de política.

La selección de variables se hizo con los mismos datos. Por eso, los valores $p$ habituales de esta regresión no incorporarían la incertidumbre de esa selección. Aquí evaluamos el modelo por sus pronósticos fuera de muestra.


## 4. ¿Qué impulsa un pronóstico Post-Lasso?

Partimos de la inflación media del entrenamiento. Para cada variable seleccionada, multiplicamos su coeficiente por la diferencia entre el valor observado y su promedio en el entrenamiento:

$$\widehat{\pi}_t=\bar{\pi}_{\mathrm{train}}+\sum_{j\in S}\widehat{\beta}_j(x_{jt}-\bar{x}_{j,\mathrm{train}}).$$

La suma de esos aportes y la media reproduce el pronóstico Post-Lasso. Un aporte negativo reduce el pronóstico respecto de la media.


In [ ]:
# Tomar los datos conocidos antes del primer mes fuera de muestra
fecha_ejemplo = df.index[inicio_oos]
valor = df.loc[fecha_ejemplo, seleccion_inicial]
promedio = X_train[seleccion_inicial].mean()

# Calcular el aporte de cada variable al pronóstico Post-Lasso
aportes = pd.DataFrame({'Valor': valor, 'Promedio': promedio,
                        'Coeficiente': coef_post})
aportes = aportes.assign(**{'Aporte (pp)':
    (aportes['Valor'] - aportes['Promedio']) * aportes['Coeficiente']})
pron_post = y_train.mean() + aportes['Aporte (pp)'].sum()

print(f'Mes pronosticado: {fecha_ejemplo:%Y-%m}')
print(f'Inflación media del entrenamiento: {y_train.mean():.3f}% mensual')
display(aportes.round(3))
print(f'Pronóstico Post-Lasso: {pron_post:.3f}% mensual')
print(f'Inflación observada: {df.loc[fecha_ejemplo, "inflacion"]:.3f}% mensual')


In [ ]:
# Graficar qué variables elevan o reducen el pronóstico Post-Lasso
if not aportes.empty:
    ax = aportes['Aporte (pp)'].plot.bar(figsize=(9, 4), color='#2ecc71')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Aporte (pp de inflación mensual)')
    ax.set_xlabel('Predictor seleccionado')
    ax.set_title(f'Aportes al pronóstico Post-Lasso: {fecha_ejemplo:%Y-%m}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 5. ¿Qué modelo pronostica mejor?

Comparamos **Ridge, Lasso, Elastic Net y Post-Lasso** con la misma muestra y el mismo esquema de la sesión anterios, es decir, ventana expansiva y reestimación cada 12 meses. En cada fecha usamos solo observaciones anteriores para entrenar. El AR(1) de inflación sirve como referencia común.

Un $R^2_{oos}$ positivo frente al AR(1) indica menor error cuadrático medio que esa referencia. Todas las métricas de error se expresan en **puntos porcentuales de inflación mensual**.


In [ ]:
# Repetir la estimación cada 12 meses con una ventana expansiva
pronosticos = []
for inicio_bloque in range(inicio_oos, len(df), 12):
    historia = df.iloc[:inicio_bloque]
    futuro = df.iloc[inicio_bloque:inicio_bloque + 12]

    # Ajustar los mismos tres modelos regularizados de U4-03
    modelos_bloque = {
        'Ridge': make_pipeline(StandardScaler(), RidgeCV(
            alphas=np.logspace(-2, 5, 50), cv=tscv)),
        'Lasso': make_pipeline(StandardScaler(), LassoCV(
            alphas=grilla, cv=tscv, max_iter=20000)),
        'Elastic Net': make_pipeline(StandardScaler(), ElasticNetCV(
            alphas=grilla, l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            cv=tscv, max_iter=20000)),
    }
    for modelo in modelos_bloque.values():
        modelo.fit(historia[X_cols], historia['inflacion'])

    # Volver a estimar por MCO las variables elegidas por Lasso
    lasso = modelos_bloque['Lasso']
    elegidas = [X_cols[j] for j in np.flatnonzero(abs(lasso[-1].coef_) > 1e-8)]
    post = ajustar_post_lasso(historia[X_cols], historia['inflacion'], elegidas)
    pred_post = (post.predict(sm.add_constant(futuro[elegidas], has_constant='add'))
                 if post is not None else np.repeat(historia['inflacion'].mean(), len(futuro)))

    # Estimar el AR(1) como referencia, con la misma historia
    ar1 = sm.OLS(historia['inflacion'], sm.add_constant(
        historia[['inflacion_lag1']], has_constant='add')).fit()
    pred_ar1 = ar1.predict(sm.add_constant(futuro[['inflacion_lag1']], has_constant='add'))

    # Guardar pronósticos de los cinco modelos para los mismos meses
    pronosticos.append(pd.DataFrame({
        'Observado': futuro['inflacion'], 'AR(1)': pred_ar1,
        'Ridge': modelos_bloque['Ridge'].predict(futuro[X_cols]),
        'Lasso': lasso.predict(futuro[X_cols]),
        'Elastic Net': modelos_bloque['Elastic Net'].predict(futuro[X_cols]),
        'Post-Lasso': pred_post,
    }, index=futuro.index))

eval_df = pd.concat(pronosticos)
display(eval_df.tail(5).round(3))


In [ ]:
# Calcular el error cuadrático medio del AR(1), nuestra referencia
error_ar1 = np.mean((eval_df['Observado'] - eval_df['AR(1)']) ** 2)

# Comparar todos los modelos en los mismos meses fuera de muestra
metricas = []
for nombre in ['AR(1)', 'Ridge', 'Lasso', 'Elastic Net', 'Post-Lasso']:
    error = eval_df['Observado'] - eval_df[nombre]
    mse = np.mean(error ** 2)
    metricas.append({'Modelo': nombre,
                     'RMSE (pp)': np.sqrt(mse),
                     'MAE (pp)': np.mean(abs(error)),
                     'R² OOS vs. AR(1)': 1 - mse / error_ar1})

tabla_metricas = pd.DataFrame(metricas).set_index('Modelo')
display(tabla_metricas.round(3))


**Cómo leer la comparación:** el modelo con menor RMSE tuvo el menor error cuadrático medio en estos meses. Compare Post-Lasso con **Ridge, Lasso y Elastic Net** en la misma tabla. La selección sin penalización puede ayudar o perjudicar: la respuesta depende de los errores fuera de muestra, no del tamaño de los coeficientes.

## 6. Conclusiones

1. Todos los predictores de precios se expresan como variaciones porcentuales mensuales. La inflación pronosticada también se expresa en porcentaje mensual.
2. Los coeficientes de Ridge, Lasso y Elastic Net describen cambios en el **pronóstico del modelo**. La estandarización determina su unidad; la penalización afecta su magnitud.
3. Post-Lasso vuelve a estimar sin penalización las variables elegidas por Lasso. Sus coeficientes tampoco representan efectos causales.
4. Para evaluar la utilidad predictiva, compare los cuatro métodos fuera de muestra bajo el mismo esquema temporal.